# Bollinger Band Reversal (BBR)

## Import Libs

In [3]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [1]:
def get_fe_price_data(
        filename: str = "../output/FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    PATH = os.getcwd()
    df = pd.read_csv(filename)
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

### Simulate Short Positions (Range-Based)

In [188]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] >= (df["Close"] + df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            win = 1
            gain = df["Close"] - tp
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data

# Set new columns 
# gains_cols = [ 
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End"
#     ]

#df[pct_50_idr_cols] = df.apply(get_bear_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Simulate Long Positions (Range-Based)

In [189]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] <= (df["Close"] - df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            win = 1
            gain = tp - df["Close"]
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data

# Set new columns 
# long_gains_cols = [
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End",
#     ]

#df[pct_50_idr_cols] = df.apply(get_bull_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Simulate Long/Short Trend Position

In [132]:
def trend_gains(
        df: Series, 
        close: Series,
        sma: Series,
        signal_name: str,
        trade_direction: int
        ):
    """Get the pip gain and apply to df
    
    - trade_direction must be:
        - Long: 1
        - Short -1
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_window = sma.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if trade_direction == 1:
                exit_condition = close_window.iloc[i] < sma_window.iloc[i]
            elif trade_direction == -1:
                exit_condition = close_window.iloc[i] > sma_window.iloc[i]
            # get stop loss time:
            if exit_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if trade_direction == 1:
            gain = close[sl_ts] - df["Close"]
        elif trade_direction == -1:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



In [64]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Trade Stats

In [133]:
def trade_stats(df: DataFrame, signal_name: str):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips

    stats = {
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips
    }

    return stats

## Breakout

In [331]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bo_df = get_fe_price_data(filename=path)
bear_bo_df = get_fe_price_data(filename=path)

long_signal = "BBU_BO"
short_signal = "BBL_BO"

# long
bull_bo_df[gains_cols] = bull_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bull_bo_df["Close"], bull_bo_df["SMA8"], long_signal, 1],
    result_type='expand'
)
# short
bear_bo_df[gains_cols] = bear_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bear_bo_df["Close"], bear_bo_df["SMA8"], short_signal, -1],
    result_type='expand'
)

bull_bo_gains_df = trade_stats(bull_bo_df.query("SMA_Trend in [0,2] and Iday_Range < (Yday_High - Yday_Low) * 0.50"), long_signal)
bear_bo_gains_df = trade_stats(bear_bo_df.query("SMA_Trend in [0,-2] and Iday_Range < (Yday_High - Yday_Low) * 0.50"), short_signal)

bo_gains_df = pd.DataFrame(data=[bull_bo_gains_df, bear_bo_gains_df],
             index=[long_signal,short_signal]
             )

bo_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
BBU_BO,48,105,153,31.372549,0.001271,-0.000707,0.060990,-0.074210,-0.01322
BBL_BO,66,95,161,40.993789,0.001864,-0.000708,0.123015,-0.067235,0.05578


In [329]:
bull_bo_df[[*gains_cols, "Close_Pct_SMA", "RSI_DVG", "SMA32_Slope_SMA", "SMA16_Slope_SMA"]].query("Win > 0").iloc[0:100]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Close_Pct_SMA,RSI_DVG,SMA32_Slope_SMA,SMA16_Slope_SMA
Date,,,,,,,,,,,
2024-03-14 03:00:00-04:00,1.0,0.0,0.000471,-0.000471,0.001365,2024-03-14 03:15:00-04:00,2024-03-14 06:30:00-04:00,0.052402,NaN,-6.749566,-14.752260
2024-03-14 04:00:00-04:00,1.0,0.0,0.000696,-0.000696,0.000315,2024-03-14 04:15:00-04:00,2024-03-14 06:30:00-04:00,0.109442,NaN,-5.638673,-3.756310
2024-03-15 05:30:00-04:00,1.0,0.0,0.000781,-0.000781,0.000105,2024-03-15 05:45:00-04:00,2024-03-15 07:00:00-04:00,0.056736,NaN,-6.116460,9.318714
2024-03-20 10:00:00-04:00,1.0,0.0,0.000757,-0.000757,0.007865,2024-03-20 10:15:00-04:00,2024-03-20 16:45:00-04:00,0.098321,NaN,-28.773365,-27.065973
2024-03-20 14:15:00-04:00,1.0,0.0,0.001530,-0.001530,0.004825,2024-03-20 14:30:00-04:00,2024-03-20 16:45:00-04:00,0.166645,NaN,8.790106,39.127460
2024-03-20 14:30:00-04:00,1.0,0.0,0.002523,-0.002523,0.002480,2024-03-20 14:45:00-04:00,2024-03-20 16:45:00-04:00,0.324947,NaN,12.992826,41.645734
2024-03-20 14:45:00-04:00,1.0,0.0,0.002867,-0.002867,0.000565,2024-03-20 15:00:00-04:00,2024-03-20 16:45:00-04:00,0.438490,NaN,17.429682,44.505556
2024-03-22 08:15:00-04:00,1.0,0.0,0.001097,-0.001097,0.000350,2024-03-22 08:30:00-04:00,2024-03-22 10:00:00-04:00,0.156041,True,-52.459923,-52.398110
2024-03-24 21:15:00-04:00,1.0,0.0,0.000561,-0.000561,0.000130,2024-03-24 21:30:00-04:00,2024-03-24 23:15:00-04:00,0.084169,NaN,-9.884751,6.975655


## SMA Breakout

In [186]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_sma_bo_df = get_fe_price_data(filename=path)
bear_sma_bo_df = get_fe_price_data(filename=path)

long_signal = "Bull_SMA_BO"
short_signal = "Bear_SMA_BO"

# long
bull_sma_bo_df[gains_cols] = bull_sma_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bull_sma_bo_df["Close"], bull_sma_bo_df["SMA4"], long_signal, 1],
    result_type='expand'
)
# short
bear_sma_bo_df[gains_cols] = bear_sma_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bear_sma_bo_df["Close"], bear_sma_bo_df["SMA4"], short_signal, -1],
    result_type='expand'
)

bull_sma_bo_gains_df = trade_stats(bull_sma_bo_df, long_signal)
bear_sma_bo_gains_df = trade_stats(bear_sma_bo_df, short_signal)

sma_bo_gains_df = pd.DataFrame(data=[bull_sma_bo_gains_df, bear_sma_bo_gains_df],
             index=[long_signal,short_signal]
             )

sma_bo_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,62,72,134,46.268657,0.001410,-0.000623,0.087405,-0.044855,0.04255
Bear_SMA_BO,57,95,152,37.500000,0.001299,-0.000639,0.074015,-0.060665,0.01335


## Bollinger Band Reversals

In [229]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bbr_df = get_fe_price_data(filename=path)
bear_bbr_df = get_fe_price_data(filename=path)

long_signal = "Bull_BBR_V2"
short_signal = "Bear_BBR_V2"

# long
bull_bbr_df[gains_cols] = bull_bbr_df.apply(
    range_long_gains,
    axis=1,
    args=[bull_bbr_df["High"],bull_bbr_df["Low"],bull_bbr_df["Close"], 
          long_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)
# short
bear_bbr_df[gains_cols] = bear_bbr_df.apply(
    range_short_gains,
    axis=1,
    args=[bear_bbr_df["High"],bear_bbr_df["Low"],bear_bbr_df["Close"], 
          short_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)

bull_bbr_gains_df = trade_stats(bull_bbr_df, long_signal)
bear_bbr_gains_df = trade_stats(bear_bbr_df, short_signal)

bbr_gains_df = pd.DataFrame(data=[bull_bbr_gains_df, bear_bbr_gains_df],
             index=[long_signal,short_signal]
             )

bbr_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BBR_V2,183,309,492,37.195122,0.002349,-0.001330,0.544886,-0.344341,0.200544
Bear_BBR_V2,150,354,504,29.761905,0.002351,-0.001313,0.458520,-0.404528,0.053992


In [215]:
bull_bbr_df[[*gains_cols, "Bull_BBR_C1","Bull_BBR_C2","Bull_BBR_C3","Bull_BBR_C4"]].nlargest(10, "Gain")

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Bull_BBR_C1,Bull_BBR_C2,Bull_BBR_C3,Bull_BBR_C4
Date,,,,,,,,,,,
2025-02-06 07:45:00-05:00,1.0,0.0,0.007922,-0.004753,0.007922,2025-02-06 08:00:00-05:00,2025-02-06 16:45:00-05:00,NaN,NaN,NaN,True
2024-08-05 02:15:00-04:00,1.0,0.0,0.007772,-0.004663,0.007772,2024-08-05 02:30:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2025-01-20 20:00:00-05:00,0.0,1.0,0.009031,-0.005419,0.007675,2025-01-20 20:15:00-05:00,2025-01-21 16:45:00-05:00,NaN,NaN,True,NaN
2025-02-06 07:30:00-05:00,1.0,0.0,0.007637,-0.004582,0.007637,2025-02-06 07:45:00-05:00,2025-02-06 16:45:00-05:00,NaN,NaN,NaN,True
2025-02-12 09:00:00-05:00,1.0,0.0,0.007403,-0.004442,0.007403,2025-02-12 09:15:00-05:00,2025-02-12 16:45:00-05:00,NaN,NaN,True,NaN
2025-02-12 08:45:00-05:00,1.0,0.0,0.007147,-0.004288,0.007147,2025-02-12 09:00:00-05:00,2025-02-12 16:45:00-05:00,NaN,NaN,True,NaN
2024-08-05 02:00:00-04:00,1.0,0.0,0.006412,-0.003847,0.006412,2024-08-05 02:15:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2024-11-25 19:15:00-05:00,1.0,0.0,0.005909,-0.003546,0.005909,2024-11-25 19:30:00-05:00,2024-11-26 16:45:00-05:00,True,NaN,NaN,True
2025-01-15 02:15:00-05:00,1.0,0.0,0.005906,-0.003544,0.005906,2025-01-15 02:30:00-05:00,2025-01-15 16:45:00-05:00,True,NaN,NaN,NaN


## Breakout Momentum

In [365]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bm_df = get_fe_price_data(filename=path)
bear_bm_df = get_fe_price_data(filename=path)

long_signal = "Bull_BM"
short_signal = "Bear_BM"

# long
bull_bm_df[gains_cols] = bull_bm_df.apply(
    range_long_gains,
    axis=1,
    args=[bull_bm_df["High"],bull_bm_df["Low"],bull_bm_df["Close"], 
          long_signal, 1, 1.25],
    result_type='expand',
    range_type="ATR4"
)
# short
bear_bm_df[gains_cols] = bear_bm_df.apply(
    range_short_gains,
    axis=1,
    args=[bear_bm_df["High"],bear_bm_df["Low"],bear_bm_df["Close"], 
          short_signal, 1, 1.25],
    result_type='expand',
    range_type="ATR4"
)

bull_bm_gains_df = trade_stats(bull_bm_df.query("Iday_Range < (Yday_High - Yday_Low) * 1.25"), long_signal)
bear_bm_gains_df = trade_stats(bear_bm_df.query("Iday_Range < (Yday_High - Yday_Low) * 1.25"), short_signal)

bo_momentum_gains_df = pd.DataFrame(data=[bull_bm_gains_df, bear_bm_gains_df],
             index=[long_signal,short_signal]
             )

bo_momentum_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BM,510,625,1135,44.933921,0.001109,-0.000937,0.595353,-0.559687,0.035665
Bear_BM,379,474,853,44.431419,0.001188,-0.000958,0.472972,-0.433770,0.039202
